# Marker expression panels (Supplementary Figure 2 / Figure 2C / S3E)

Heatmaps show expression of selected marker genes for coarse cell types (S2A), CD8 T cell subsets (S2B), myeloid subsets (S2C) scaled by gene, or the ratio of infected/uninfected expression for myeloid subsets (S2D). The CD8 T cell panel shows data from infected samples and includes populations expanded during infection, with naïve CD8 T cells included for comparison. The myeloid and coarse cluster panels show data from both infected and uninfected samples. Bold text indicates genes that are part of the Hallmark interferon-γ response. Violin plots of Hallmark interferon-γ response enrichment scores for infected vs uninfected cells are shown for coarse and myeloid clusters (S2E,F).

Visium niche markers and focus genes (including Cxcl9, Cxcl10, Ifng, and Il27) relate to Fig. 2C / S3E and to IL27 as discussed in the text.

| Panel | Content |
|-------|---------|
| S2A | Coarse lineages |
| S2B | CD8 fine types (infected) |
| S2C–D | Myeloid fine types (scaled / infected÷uninfected) |
| S2E–F | Hallmark IFN-γ enrichment scores |
| Fig. 2C / S3E | Visium niche markers + focus genes |



## 1. Setup


In [ ]:
from pathlib import Path
import os
from collections import OrderedDict

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy import sparse
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Paths (override with env vars or edit here)
PATH_SCRNA = Path(os.environ.get("SCRNA_H5AD", "data/scrna_spleen_atlas.h5ad"))
PATH_C2L_INF = Path(os.environ.get("C2L_INFECTED_H5AD", "data/cell2location_infected.h5ad"))
PATH_C2L_UNINF = Path(os.environ.get("C2L_UNINFECTED_H5AD", "data/cell2location_uninfected.h5ad"))


def _split_paths(env_key, default):
    raw = os.environ.get(env_key)
    if raw:
        return [Path(p) for p in raw.split(":") if p]
    return [Path(p) for p in default]


PATH_VISIUM_INF = _split_paths(
    "VISIUM_INFECTED_H5",
    [
        "data/visium/V1S1_infected/filtered_feature_bc_matrix.h5",
        "data/visium/V1S2_infected/filtered_feature_bc_matrix.h5",
    ],
)
PATH_VISIUM_UNINF = _split_paths(
    "VISIUM_UNINFECTED_H5",
    [
        "data/visium/V1S3_uninfected/filtered_feature_bc_matrix.h5",
        "data/visium/V1S4_uninfected/filtered_feature_bc_matrix.h5",
    ],
)

OUT_DIR = Path(os.environ.get("EXPRESSION_OUT", "outputs"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

FINE_KEY = "celltypes_redo"
COARSE_KEY = "coarse_redo"
TIME_KEY = "timepoint"
CLUSTER_KEY = "paper_clusters"


def truncate_cmap(name, lo=0.10, hi=0.9, n=256):
    base = plt.get_cmap(name)
    return LinearSegmentedColormap.from_list(
        f"{name}_trunc", base(np.linspace(lo, hi, n))
    )


SOFT_RDBU = truncate_cmap("RdBu_r", lo=0.10, hi=0.9)


def dense_X(adata):
    X = adata.X
    return X.toarray() if sparse.issparse(X) else np.asarray(X)


def scale_cols_01(df):
    out = df.copy().astype(float)
    for g in out.columns:
        col = out[g].values
        lo, hi = col.min(), col.max()
        out[g] = (col - lo) / (hi - lo) if hi > lo else 0.0
    return out


def mean_expression_by_category(adata, obs_key, categories, genes, label_map=None):
    sub = adata[:, genes].copy()
    rows, labels = [], []
    for cat in categories:
        m = sub.obs[obs_key] == cat
        if not m.any():
            continue
        rows.append(dense_X(sub[m]).mean(axis=0))
        labels.append(label_map[cat] if label_map else cat)
    return pd.DataFrame(np.vstack(rows), index=labels, columns=genes)


def order_genes_by_peak(S, row_order):
    by_peak = {ct: [] for ct in row_order}
    for gene in S.columns:
        peak = S[gene].idxmax()
        if peak in by_peak:
            by_peak[peak].append(gene)
    return [g for ct in row_order for g in by_peak[ct]]


def save_heatmap(fig, stem):
    fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(OUT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    print("Saved", OUT_DIR / f"{stem}.pdf")


adata_sc = sc.read_h5ad(PATH_SCRNA)
print(adata_sc.n_obs, "cells |", adata_sc.n_vars, "genes")


## 1b. Publication fonts / gene-label italics


In [ ]:
import matplotlib as mpl
from matplotlib.font_manager import FontProperties

PUB_STYLE = {
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}
mpl.rcParams.update(PUB_STYLE)

TITLE_FP = FontProperties(family="Arial", weight="bold", size=14)
XTICK_FP = FontProperties(family="Arial", style="normal", size=10)
YTICK_FP = FontProperties(family="Arial", size=12)
GENE_FP = FontProperties(family="Arial", style="italic", size=12)
CBAR_FP = FontProperties(family="Arial", weight="bold", size=10)
CBAR_TICK_FP = FontProperties(family="Arial", size=8)


def _italicize_gene_labels(fig):
    """Italicize rotated x-tick labels (gene names) on every axes."""
    for ax in fig.axes:
        if ax.get_label() == "<colorbar>" or getattr(ax, "_colorbar", False):
            continue
        xlabels = ax.get_xticklabels()
        if not xlabels or not str(xlabels[0].get_text()).strip():
            continue
        rot = xlabels[0].get_rotation()
        try:
            rot = abs(float(rot))
        except Exception:
            rot = 90.0 if str(rot).lower() == "vertical" else 0.0
        if rot < 45:
            continue
        for lbl in xlabels:
            size = lbl.get_fontsize()
            lbl.set_fontproperties(
                FontProperties(family="Arial", style="italic", size=size)
            )


def _italicize_all_open_figures():
    for num in plt.get_fignums():
        _italicize_gene_labels(plt.figure(num))


# Patch plt.show (idempotent) so gene ticks italicize on display
if not hasattr(plt, "_original_show"):
    plt._original_show = plt.show

    def _show(*args, **kwargs):
        _italicize_all_open_figures()
        return plt._original_show(*args, **kwargs)

    plt.show = _show

try:
    from IPython import get_ipython
    _ip = get_ipython()
except Exception:
    _ip = None

if _ip is not None and not getattr(_ip, "_gene_italic_hooked", False):
    _ip.events.register("post_run_cell", lambda result: _italicize_all_open_figures())
    _ip._gene_italic_hooked = True


## 2. S2A - Coarse lineages

Per-gene min-max scaled mean expression across coarse cell types (infected + uninfected).


In [ ]:
coarse_genes = [
    "Klrb1c", "Klre1", "Cd4", "Cd8a", "Cd8b1", "Trbc2", "Cd3d", "Cd3e",
    "Themis", "Cd79b", "Cd19", "Cd79a", "Gclm", "Prdx2", "Cdh5", "Stab2",
    "Col1a1", "Col1a2", "Cpa3", "Cyp11a1", "Csf3r", "Csf1r", "Cst3",
    "Camp", "Ngp", "S100a8", "S100a9",
]
coarse_types = [
    "NK", "CD4-Tcell", "CD8-Tcell", "Bcell", "Hematopoietic",
    "Endothelial", "Fibroblast", "Mast-Cell", "Myeloid", "Neutrophil",
]

adata_coarse = adata_sc[adata_sc.obs[COARSE_KEY].isin(coarse_types)].copy()
genes = [g for g in coarse_genes if g in adata_coarse.var_names]
expr = mean_expression_by_category(adata_coarse, COARSE_KEY, coarse_types, genes)
S = scale_cols_01(expr).reindex(index=coarse_types, columns=genes)

fig, ax = plt.subplots(figsize=(12, 4.5))
sns.heatmap(
    S, cmap=SOFT_RDBU, vmin=0, vmax=1, linewidths=0, ax=ax,
    cbar_kws={"label": "Scaled expression"},
)
ax.set_title("S2A - coarse lineages")
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=90, ha="center", fontstyle="italic")
plt.yticks(rotation=0)
plt.tight_layout()
save_heatmap(fig, "s2a_coarse_scaled")
plt.show()


## 3. S2B - CD8 fine types (infected only)

Per-gene scaled means; genes ordered by peak CD8 state. Includes naïve for comparison.


In [ ]:
cd8_map = {
    "CD8-Tcell_naive": "CD8-naive",
    "CD8-Tcell_early-active": "CD8-early-act",
    "CD8-Tcell_proliferating": "CD8-prolif",
    "CD8-Tcell_late-active": "CD8-late-act",
    "CD8-Tcell_effector": "CD8-eff",
}
cd8_order = list(cd8_map.values())
cd8_genes = list(dict.fromkeys([
    "Sell", "Tcf7", "Ccr7", "Lef1", "Il27ra", "Ikzf2", "Ifngas1", "Bcl2", "Eomes", "Ifng",
    "Gzmb", "Gzmk", "Gzma", "Cx3cr1", "Prf1", "Ccr5", "Tbx21", "Il18r1",
    "Il18rap", "Cd5", "Txn1", "Stmn1", "Mki67", "Cxcr5", "Klrg1", "Cd44",
    "Cxcr3", "Cd28", "Icos", "Il6st",
]))

adata_cd8 = adata_sc[
    (adata_sc.obs[TIME_KEY] == "3wk")
    & (adata_sc.obs[FINE_KEY].isin(cd8_map))
].copy()
genes = [g for g in cd8_genes if g in adata_cd8.var_names]
expr = mean_expression_by_category(
    adata_cd8, FINE_KEY, list(cd8_map.keys()), genes, label_map=cd8_map
)
S = scale_cols_01(expr).reindex(index=cd8_order)
S = S[order_genes_by_peak(S, cd8_order)]

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    S, cmap=SOFT_RDBU, vmin=0, vmax=1, linewidths=0, ax=ax,
    cbar_kws={"label": "Scaled expression"},
)
ax.set_title("S2B - CD8 fine types (infected)")
plt.xticks(rotation=90, ha="center", fontstyle="italic")
plt.yticks(rotation=0)
plt.tight_layout()
save_heatmap(fig, "s2b_cd8_scaled")
plt.show()


## 4. S2C-D - Myeloid fine types

Left: per-gene scaled expression (all timepoints). Right: log2(infected / uninfected) mean expression with detection filters. IFN-γ hallmark genes bolded on the y-axis.


In [ ]:
myeloid_map = {
    "Myeloid_pDC": "pDC",
    "Myeloid_moDC-CXCL9/10": "moDC-B",
    "Myeloid_moDC": "moDC-A",
    "Myeloid_DC2": "cDC2",
    "Myeloid_cDC1": "cDC1",
    "Myeloid_monocyte": "Monocyte",
    "Myeloid_migratory": "MigDC",
    "Myeloid_marginalzone": "Mac-F480",
    "Myeloid_Macrophage-CXCL9/10": "Mac-B",
    "Myeloid_Macrophage": "Mac-A",
}
myeloid_order = list(myeloid_map.values())
myeloid_genes = [
    "Relt", "Ccl6", "Il1b", "Csf3r", "S100a8", "S100a9", "Ccl9", "Ccr2", "Fn1",
    "Chil3", "Il27", "Ace", "Ear2", "Cxcl16", "H2-M2", "Ccl5", "Ccr7",
    "Clec4a4", "Rtn1", "Ccnd1", "Dscam", "Ifng", "Mgl2", "Mmp12", "Ccr5",
    "Siglech", "Clec9a", "Tlr3", "Xcr1", "Cxcl9", "Adgre1", "Il18", "Mrc1",
    "C1qc", "Cd5l", "Vcam1", "Slamf8", "Cxcl10", "Mmp14", "Cxcr3", "Il10",
    "Il12a", "Il12b", "Cd86", "Cd80", "Ccl19", "Ccl21",
    "Ugcg", "Ets2", "Gm2a", "Irf1", "Irf9", "Neu1",
    "Ltc4s", "Ccl24", "Vsig10", "Col23a1", "Ccl4", "Stat1", "Cd74",
]
IFNG_BOLD = {
    "Irf9", "Stat1", "Vcam1", "Ccl5", "Irf1", "Cd74", "Cxcl9", "Xcr1",
    "Cd86", "Cxcl10", "Adgre1", "Ccr7", "Rtn1",
}
DET_FRAC = 0.02
MIN_CELLS = 5

adata_mye = adata_sc[adata_sc.obs[FINE_KEY].isin(myeloid_map)].copy()
genes = [g for g in myeloid_genes if g in adata_mye.var_names]
expr = mean_expression_by_category(
    adata_mye, FINE_KEY, list(myeloid_map.keys()), genes, label_map=myeloid_map
)
S = scale_cols_01(expr).reindex(index=myeloid_order)
S = S[order_genes_by_peak(S, myeloid_order)]
genes_ordered = list(S.columns)
S_plot = S.T

inf = adata_sc[adata_sc.obs[TIME_KEY] == "3wk"].copy()
un = adata_sc[adata_sc.obs[TIME_KEY] != "3wk"].copy()
ratio = pd.DataFrame(np.nan, index=genes_ordered, columns=myeloid_order)
for raw, label in myeloid_map.items():
    mi = inf.obs[FINE_KEY] == raw
    mu = un.obs[FINE_KEY] == raw
    if not mi.any() or not mu.any():
        continue
    Xi = dense_X(inf[mi, genes_ordered])
    Xu = dense_X(un[mu, genes_ordered])
    mean_i, mean_u = Xi.mean(0), Xu.mean(0)
    det_i, det_u = (Xi > 0).mean(0), (Xu > 0).mean(0)
    n_i, n_u = (Xi > 0).sum(0), (Xu > 0).sum(0)
    for j, g in enumerate(genes_ordered):
        if (
            det_i[j] >= DET_FRAC
            and det_u[j] >= DET_FRAC
            and n_i[j] >= MIN_CELLS
            and n_u[j] >= MIN_CELLS
            and mean_u[j] > 0
        ):
            ratio.loc[g, label] = mean_i[j] / mean_u[j]

log2_ratio = np.log2(ratio.astype(float))
finite = log2_ratio.values[np.isfinite(log2_ratio.values)]
vbound = float(np.ceil(max(abs(finite.min()), abs(finite.max())))) if len(finite) else 1.0

fig, axes = plt.subplots(
    1, 2, figsize=(12, max(8, 0.22 * len(genes_ordered))), sharey=True
)
sns.heatmap(
    S_plot, cmap=SOFT_RDBU, vmin=0, vmax=1, ax=axes[0],
    cbar_kws={"label": "Scaled expression", "shrink": 0.4},
    linewidths=0.3, linecolor="white",
)
sns.heatmap(
    log2_ratio, cmap="RdBu_r", center=0, vmin=-vbound, vmax=vbound, ax=axes[1],
    cbar_kws={"label": "log2(inf/uninf)", "shrink": 0.4},
    linewidths=0.3, linecolor="white",
)
axes[0].set_title("S2C - myeloid scaled")
axes[1].set_title("S2D - myeloid inf/uninf")
for ax in axes:
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=45)
for lbl, g in zip(axes[0].get_yticklabels(), genes_ordered):
    lbl.set_fontstyle("italic")
    if g in IFNG_BOLD:
        lbl.set_fontweight("bold")
plt.tight_layout()
save_heatmap(fig, "s2cd_myeloid_scaled_and_ratio")
plt.show()


## 5. S2E-F - Hallmark IFN-γ response scores

Score cells with the MSigDB Hallmark IFN-γ response gene set (mouse symbols via `gseapy`), then violin-plot infected vs uninfected. Stars: BH-adjusted Mann-Whitney (*P* < 0.05 / 0.01 / 0.001).


In [ ]:
import gseapy as gp

SCORE_KEY = "IFNG_Hallmark_score"
PALETTE = {"Infected": "#C44E52", "Uninfected": "#4C72B0"}


def human_to_mouse_symbol(g):
    g = str(g)
    return g[0].upper() + g[1:].lower() if g else g


mouse_hallmark = gp.get_library(name="MSigDB_Hallmark_2020", organism="Mouse")
ifng_genes = [human_to_mouse_symbol(g) for g in mouse_hallmark["Interferon Gamma Response"]]
ifng_genes = [g for g in ifng_genes if g in adata_sc.var_names]
print(f"IFN-γ genes in adata: {len(ifng_genes)}")


def infection_labels(adata):
    out = adata.copy()
    out.obs["infection"] = np.where(
        out.obs[TIME_KEY].astype(str) == "3wk", "Infected", "Uninfected"
    )
    out.obs["infection"] = pd.Categorical(
        out.obs["infection"], categories=["Infected", "Uninfected"], ordered=True
    )
    return out


def mw_stats(obs, group_key, order):
    rows = []
    for cl in order:
        sub = obs[obs[group_key] == cl]
        a = sub.loc[sub["infection"] == "Infected", SCORE_KEY].dropna().values
        b = sub.loc[sub["infection"] == "Uninfected", SCORE_KEY].dropna().values
        p = (
            mannwhitneyu(a, b, alternative="two-sided").pvalue
            if len(a) >= 10 and len(b) >= 10
            else np.nan
        )
        rows.append({"cluster": cl, "pval": p, "n_inf": len(a), "n_un": len(b)})
    stats = pd.DataFrame(rows)
    ok = stats["pval"].notna()
    stats["padj"] = np.nan
    if ok.any():
        stats.loc[ok, "padj"] = multipletests(stats.loc[ok, "pval"], method="fdr_bh")[1]
    stats["sig"] = pd.cut(
        stats["padj"],
        bins=[-np.inf, 0.001, 0.01, 0.05, np.inf],
        labels=["***", "**", "*", "ns"],
    )
    return stats


def plot_ifng_violin(obs, group_key, order, title, stem):
    fig, ax = plt.subplots(figsize=(10, 5.2))
    sns.violinplot(
        data=obs,
        x=group_key,
        y=SCORE_KEY,
        hue="infection",
        hue_order=["Infected", "Uninfected"],
        palette=PALETTE,
        order=order,
        cut=0,
        inner="quartile",
        linewidth=0.7,
        dodge=True,
        width=0.75,
        density_norm="width",
        ax=ax,
    )
    stats = mw_stats(obs, group_key, order)
    print(title)
    print(stats.to_string(index=False))
    y_max = obs[SCORE_KEY].max()
    ax.set_ylim(-0.06, y_max * 1.28)
    sig_map = stats.set_index("cluster")["sig"].astype(str).to_dict()
    for i, cl in enumerate(order):
        s = sig_map.get(cl, "ns")
        if s != "ns":
            ax.text(i, y_max * 1.06, s, ha="center", va="bottom", fontsize=16, fontweight="bold")
    ax.set_title(title, fontsize=16, fontweight="bold", pad=12)
    ax.set_xlabel("")
    ax.set_ylabel("IFN-γ Hallmark score")
    ax.tick_params(axis="x", rotation=90)
    ax.legend(title="", frameon=False, loc="upper right")
    sns.despine(ax=ax)
    plt.tight_layout()
    save_heatmap(fig, stem)
    plt.show()
    return stats


# S2E - coarse
COARSE_DISPLAY = OrderedDict([
    ("Bcell", "B Cell"),
    ("CD4-Tcell", "CD4 T Cell"),
    ("CD8-Tcell", "CD8 T Cell"),
    ("Myeloid", "Myeloid"),
    ("NK", "NK"),
    ("Neutrophil", "Neutrophil"),
    ("Fibroblast", "Fibroblast"),
    ("Endothelial", "Endothelial"),
    ("Hematopoietic", "Hematopoietic"),
    ("Mast-Cell", "Mast Cell"),
])
COARSE_ORDER = list(COARSE_DISPLAY.values())

ad_c = infection_labels(adata_sc)
ad_c.obs[COARSE_KEY] = ad_c.obs[COARSE_KEY].astype(str).replace({"Hematopoeitic": "Hematopoietic"})
ad_c.obs["coarse_label"] = ad_c.obs[COARSE_KEY].map(COARSE_DISPLAY)
ad_c = ad_c[ad_c.obs["coarse_label"].isin(COARSE_ORDER)].copy()
ad_c.obs["coarse_label"] = pd.Categorical(ad_c.obs["coarse_label"], categories=COARSE_ORDER, ordered=True)
sc.tl.score_genes(ad_c, gene_list=ifng_genes, score_name=SCORE_KEY, use_raw=False)
stats_s2e = plot_ifng_violin(
    ad_c.obs, "coarse_label", COARSE_ORDER,
    "S2E - Hallmark IFN-γ score (coarse)", "s2e_ifng_violin_coarse",
)
stats_s2e.to_csv(OUT_DIR / "s2e_ifng_stats_coarse.csv", index=False)

# S2F - myeloid fine
MYELOID_DISPLAY_IFN = OrderedDict([
    ("Myeloid_cDC1", "cDC1"),
    ("Myeloid_DC2", "cDC2"),
    ("Myeloid_migratory", "MigDC"),
    ("Myeloid_moDC", "moDC-A"),
    ("Myeloid_moDC-CXCL9/10", "moDC-B"),
    ("Myeloid_monocyte", "Monocyte"),
    ("Myeloid_Macrophage", "Mac-A"),
    ("Myeloid_Macrophage-CXCL9/10", "Mac-B"),
    ("Myeloid_marginalzone", "Mac-F480"),
    ("Myeloid_pDC", "pDC"),
])
MYE_ORDER = list(MYELOID_DISPLAY_IFN.values())

ad_m = infection_labels(adata_sc)
ad_m.obs["cell_type"] = ad_m.obs[FINE_KEY].astype(str).map(MYELOID_DISPLAY_IFN)
ad_m = ad_m[ad_m.obs["cell_type"].isin(MYE_ORDER)].copy()
ad_m.obs["cell_type"] = pd.Categorical(ad_m.obs["cell_type"], categories=MYE_ORDER, ordered=True)
sc.tl.score_genes(ad_m, gene_list=ifng_genes, score_name=SCORE_KEY, use_raw=False)
stats_s2f = plot_ifng_violin(
    ad_m.obs, "cell_type", MYE_ORDER,
    "S2F - Hallmark IFN-γ score (myeloid)", "s2f_ifng_violin_myeloid",
)
stats_s2f.to_csv(OUT_DIR / "s2f_ifng_stats_myeloid.csv", index=False)


## 6. Focus genes - *Cxcl9* / *Cxcl10* / *Il27* (myeloid)

Compact myeloid heatmaps for the main infection-induced ligands highlighted in the text (scaled expression + log2 infected/uninfected ratio).


In [ ]:
FOCUS_MYE = ["Cxcl9", "Cxcl10", "Il27"]
# reuse myeloid_map / myeloid_order from S2C-D if present; else redefine
if "myeloid_map" not in globals():
    myeloid_map = {
        "Myeloid_pDC": "pDC",
        "Myeloid_moDC-CXCL9/10": "moDC-B",
        "Myeloid_moDC": "moDC-A",
        "Myeloid_DC2": "cDC2",
        "Myeloid_cDC1": "cDC1",
        "Myeloid_monocyte": "Monocyte",
        "Myeloid_migratory": "MigDC",
        "Myeloid_marginalzone": "Mac-F480",
        "Myeloid_Macrophage-CXCL9/10": "Mac-B",
        "Myeloid_Macrophage": "Mac-A",
    }
    myeloid_order = list(myeloid_map.values())

genes_use = [g for g in FOCUS_MYE if g in adata_sc.var_names]
adata_mye = adata_sc[adata_sc.obs[FINE_KEY].isin(myeloid_map)].copy()
expr = mean_expression_by_category(
    adata_mye, FINE_KEY, list(myeloid_map.keys()), genes_use, label_map=myeloid_map
)
S_focus = scale_cols_01(expr).reindex(index=myeloid_order, columns=genes_use)

inf = adata_sc[adata_sc.obs[TIME_KEY] == "3wk"].copy()
un = adata_sc[adata_sc.obs[TIME_KEY] != "3wk"].copy()
ratio = pd.DataFrame(np.nan, index=myeloid_order, columns=genes_use)
for raw, label in myeloid_map.items():
    mi = inf.obs[FINE_KEY] == raw
    mu = un.obs[FINE_KEY] == raw
    if not mi.any() or not mu.any():
        continue
    Xi = dense_X(inf[mi, genes_use])
    Xu = dense_X(un[mu, genes_use])
    mean_i, mean_u = Xi.mean(0), Xu.mean(0)
    det_i, det_u = (Xi > 0).mean(0) * 100, (Xu > 0).mean(0) * 100
    n_i, n_u = (Xi > 0).sum(0), (Xu > 0).sum(0)
    for j, g in enumerate(genes_use):
        if det_i[j] >= 2 and det_u[j] >= 2 and n_i[j] >= 5 and n_u[j] >= 5 and mean_u[j] > 0:
            ratio.loc[label, g] = mean_i[j] / mean_u[j]
log2_focus = np.log2(ratio.astype(float))
finite = log2_focus.values[np.isfinite(log2_focus.values)]
vbound = float(np.ceil(max(abs(finite.min()), abs(finite.max())))) if len(finite) else 1.0

fig, axes = plt.subplots(1, 2, figsize=(7.5, 4.5), sharey=True)
sns.heatmap(
    S_focus, cmap=SOFT_RDBU, vmin=0, vmax=1, ax=axes[0],
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Scaled expression", "shrink": 0.7},
)
sns.heatmap(
    log2_focus, cmap="RdBu_r", center=0, vmin=-vbound, vmax=vbound, ax=axes[1],
    linewidths=0.5, linecolor="white", mask=~np.isfinite(log2_focus),
    cbar_kws={"label": "log2(inf/uninf)", "shrink": 0.7},
)
axes[0].set_title("Myeloid - scaled")
axes[1].set_title("Myeloid - ratio")
for ax in axes:
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels(genes_use, rotation=90, fontstyle="italic")
plt.tight_layout()
save_heatmap(fig, "focus_cxcl9_cxcl10_il27_myeloid")
plt.show()


## 7. Fig. 2C / S3E - Visium niche markers

Matches Space Ranger barcodes to `paper_clusters` on the cell2location object, then plots z-scored mean log-normalized expression per niche.


In [ ]:
CLUSTER_MAP = {
    "TZ": "WP-A: TZ",
    "BZ": "WP-B: BZ",
    "2nd Fol": "WP-D: GC",
    "MZ": "WP-C: MZ",
    "RP-A-PC": "RP-A: GZMK",
    "RP-B-MK": "RP-B: H_MK",
    "RP-C-Neut": "RP-C: NGP",
    "RP-D": "RP-D: F480",
    "RP-E-RBC": "RP-E: Rhag",
}
obs_to_short = {v: k for k, v in CLUSTER_MAP.items()}

manual_gene_lists = OrderedDict([
    ("TZ", ["Ccl21a", "Ccl19", "Trbc1", "Ccl5", "Thy1", "Cxcl16"]),
    ("BZ", ["Ly6d", "Iglc2", "Cd38", "Blk"]),
    ("2nd Fol", ["Pou2af1", "Neil1", "Cxcl13", "Aicda", "Cd74"]),
    ("MZ", ["Gm2a", "Irf1", "Cxcl10", "Cxcl9", "Neu1", "Ltc4s", "Vsig10", "Col23a1", "Rtn1", "Mmp12", "Stat1"]),
    ("RP-A-PC", ["Igha", "Jchain", "Il27"]),
    ("RP-B-MK", ["Ppbp", "Pf4", "Cxcl12", "Thbs1"]),
    ("RP-C-Neut", ["S100a9", "Ngp", "Camp"]),
    ("RP-D", ["Hmox1", "Adgre1", "Ugcg", "Ets2", "Ifng", "Gbp4"]),
    ("RP-E-RBC", ["Slc4a1", "Apol11b", "Alas2"]),
])
ROW_ORDER = list(manual_gene_lists.keys())
GENES = [g for genes in manual_gene_lists.values() for g in genes]


def match_visium_to_clusters(c2l_path, visium_paths):
    c2l = sc.read_h5ad(c2l_path)
    matrices = []
    for p in visium_paths:
        a = sc.read_10x_h5(p)
        a.var_names_make_unique()
        matrices.append(a)

    parsed = [
        (bc, bc.split("_")[0], "_".join(bc.split("_")[1:])) for bc in c2l.obs_names
    ]
    suffixes = {s for _, _, s in parsed}
    name_sets = [set(a.obs_names) for a in matrices]

    suffix_to_idx = {}
    for suf in suffixes:
        bases = {b for _, b, s in parsed if s == suf}
        overlaps = [len(bases & s) for s in name_sets]
        suffix_to_idx[suf] = int(np.argmax(overlaps))

    parts = []
    matched_c2l = []
    for c2l_bc, base, suf in parsed:
        i = suffix_to_idx.get(suf)
        if i is None or base not in matrices[i].obs_names:
            continue
        matched_c2l.append(c2l_bc)
        part = matrices[i][[base]].copy()
        part.obs_names = pd.Index([c2l_bc])
        parts.append(part)

    adata = sc.concat(parts, join="outer", fill_value=0)
    adata.obs[CLUSTER_KEY] = c2l[matched_c2l].obs[CLUSTER_KEY].values
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    return adata


def niche_zscore_matrix(adata):
    genes = [g for g in GENES if g in adata.var_names]
    short = adata.obs[CLUSTER_KEY].map(obs_to_short)
    rows = []
    for niche in ROW_ORDER:
        m = short == niche
        if not m.any():
            rows.append(np.full(len(genes), np.nan))
            continue
        rows.append(dense_X(adata[m, genes]).mean(0))
    mat = pd.DataFrame(rows, index=ROW_ORDER, columns=genes)
    return mat.apply(
        lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) else 0.0, axis=0
    )


adata_vis_inf = match_visium_to_clusters(PATH_C2L_INF, PATH_VISIUM_INF)
Z_inf = niche_zscore_matrix(adata_vis_inf)

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(Z_inf, cmap="RdBu_r", center=0, ax=ax, cbar_kws={"label": "Z-score"})
ax.set_title("Fig. 2C - Visium niche markers (infected)")
plt.xticks(rotation=90, ha="center", fontstyle="italic")
plt.yticks(rotation=0)
plt.tight_layout()
save_heatmap(fig, "fig2c_visium_niche_markers_infected")
plt.show()

if PATH_C2L_UNINF.exists() and all(p.exists() for p in PATH_VISIUM_UNINF):
    adata_vis_un = match_visium_to_clusters(PATH_C2L_UNINF, PATH_VISIUM_UNINF)
    Z_un = niche_zscore_matrix(adata_vis_un)
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    sns.heatmap(Z_un, cmap="RdBu_r", center=0, ax=axes[0], cbar_kws={"label": "Z-score"})
    sns.heatmap(Z_inf, cmap="RdBu_r", center=0, ax=axes[1], cbar_kws={"label": "Z-score"})
    axes[0].set_title("S3E - uninfected")
    axes[1].set_title("S3E - infected")
    for ax in axes:
        ax.tick_params(axis="x", rotation=90)
        for lbl in ax.get_xticklabels():
            lbl.set_fontstyle("italic")
    plt.tight_layout()
    save_heatmap(fig, "s3e_visium_niche_markers_both")
    plt.show()
else:
    print("Skipping S3E uninfected panel (paths missing).")


## 8. Focus genes - Visium niches (*Cxcl9* / *Cxcl10* / *Ifng* / *Il27*)

Per-niche scaled mean expression on infected Visium (reuses the matched object from section 5 when available).


In [ ]:
FOCUS_VIS = ["Cxcl9", "Cxcl10", "Ifng", "Il27"]

if "adata_vis_inf" not in globals():
    adata_vis_inf = match_visium_to_clusters(PATH_C2L_INF, PATH_VISIUM_INF)

genes = [g for g in FOCUS_VIS if g in adata_vis_inf.var_names]
short = adata_vis_inf.obs[CLUSTER_KEY].map(obs_to_short)
rows, labels = [], []
for niche in ROW_ORDER:
    m = short == niche
    if not m.any():
        continue
    rows.append(dense_X(adata_vis_inf[m, genes]).mean(0))
    labels.append(niche)
E = pd.DataFrame(rows, index=labels, columns=genes)
S = scale_cols_01(E)
print(E.round(3))

fig, ax = plt.subplots(figsize=(5.5, 4.2))
sns.heatmap(
    S, cmap=SOFT_RDBU, vmin=0, vmax=1, ax=ax,
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Scaled expression"},
)
ax.set_title("Visium focus genes - infected")
ax.set_xticklabels(genes, rotation=90, fontstyle="italic")
ax.set_yticklabels(labels, rotation=0)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
save_heatmap(fig, "focus_cxcl9_cxcl10_ifng_il27_visium")
plt.show()


## Notes

- Source: `paper_heatmaps_submission_040626_YK.ipynb` (not edited).
- Obs keys: `celltypes_redo`, `coarse_redo`, `timepoint` on scRNA; `paper_clusters` on cell2location.
- S2E-F needs `gseapy` (Hallmark IFN-γ gene set download on first run).
- Section 1b applies Arial-first rcParams, PDF fonttype 42, and auto-italicizes rotated gene tick labels on `plt.show` / cell display (same approach as the submission notebook).
